# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 03. 검정·모델링용 데이터 가공 — SQL 재현
- 목표: PY_03·PY_04에서 검정과 모델링 전에 pandas로 만든 변수(광역시·강원더미·보험자부담률·순위·Q3 기준 고진료비)를 SQL로 다시 만들고, 결과가 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (sql_practice.db의 hira_eda 테이블 사용)
- 대조 기준: PY_03_Statistical_Testing.ipynb, PY_04_Modeling.ipynb 실행 결과
- 범위 밖: 정규성·Mann-Whitney·Kruskal-Wallis·Dunn 검정, 회귀 적합 (Python 분석 영역). 단, 마지막에 SQL로 만든 테이블을 Python 모델에 넣어 결과가 같은지 확인한다.

### 3.1 환경 설정
#### 3.1-1 DB 연결 및 저장된 테이블 확인

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira
1,hira_eda


#### 3.1-2 SQLite 버전·수학 함수 사용 가능 여부 확인
- 윈도우 함수(3.3)는 SQLite 3.25 이상에서만 동작하므로 버전을 확인한다
- SQLite는 설치 방식에 따라 제곱근(`sqrt`)·자연로그(`ln`) 함수가 없을 수 있다 → 함수별로 따로 확인
- `try / except`: 에러가 나도 셀이 멈추지 않고 "사용 불가"로 출력한 뒤 다음 함수로 넘어간다

In [2]:
print(pd.read_sql("SELECT sqlite_version() AS 버전", conn))

for func in ['sqrt(16)', 'ln(10)']:
    try:
        r = pd.read_sql(f"SELECT {func} AS 값", conn)
        print(func, '→ 사용 가능:', r.iloc[0, 0])
    except Exception as e:
        print(func, '→ 사용 불가:', e)

       버전
0  3.51.0
sqrt(16) → 사용 불가: Execution failed on sql 'SELECT sqrt(16) AS 값': no such function: sqrt
ln(10) → 사용 불가: Execution failed on sql 'SELECT ln(10) AS 값': no such function: ln


> **SQLite 3.51.0 — 윈도우 함수 사용 가능, 수학 함수(sqrt·ln)는 없음** <br>
> 버전 3.51.0으로 윈도우 함수 사용 조건(3.25 이상)을 충족. <br>
> `sqrt`, `ln` 모두 "no such function" → 표준편차(제곱근 필요)와 로그 변환은 SQL이 아닌 Python에서 계산한다.

### 3.2 그룹 변수·비율 변수 생성
#### 3.2-1 모델링용 테이블 만들기 (광역시·강원더미·보험자부담률)
- PY_03 광역시 변수, PY_04 강원더미, PY_03 3.6 보험자부담률 대응 (SAS `IF 시도 IN (...) THEN 광역시 = 1` 대응)
- `CASE WHEN 조건 THEN 1 ELSE 0 END`: 조건을 만족하면 1, 아니면 0인 열을 만든다
- 테이블이 단계별로 쌓이는 구조: hira(원자료) → hira_eda(파생변수) → hira_model(그룹·비율 변수)

In [4]:
conn.execute("DROP TABLE IF EXISTS hira_model")
conn.execute("""
CREATE TABLE hira_model AS
SELECT *,
       CASE WHEN 시도 IN ('서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종')
            THEN 1 ELSE 0 END AS 광역시,
        CASE WHEN 시도 = '강원' THEN 1 ELSE 0 END AS 강원더미,
        ROUND(CAST(보험자부담금 AS REAL) / 요양급여비용총액 * 100, 1) AS 보험자부담률
FROM hira_eda
""")
conn.commit()

#### 3.2-2 광역시 여부별 시군구 수·평균
- PY_03 [검정 2] 그룹별 기술통계 중 n·평균 대응 (중앙값·표준편차는 3.4에서)

In [5]:
q = """
SELECT 광역시,
      COUNT(*) AS n,
      ROUND(AVG(일인당_진료비_만원), 1) AS 평균_만원
FROM hira_model
GROUP BY 광역시
"""
pd.read_sql(q, conn)

,광역시,n,평균_만원
0,0,175,72.0
1,1,76,82.0


> **광역시 여부별 n·평균 — PY_03과 일치** <br>
> 광역시(1): 76개, 평균 82.0만원 / 비광역시(0): 175개, 평균 72.0만원.

#### 3.2-3 보험자부담률 상위 5개 시군구
- PY_02 해석 문장 "화순군 보험자부담률 84.9%로 전국 최고" 검증 (코드 출력으로는 확인한 적 없는 수치)

In [7]:
q = """
SELECT 시도, 시군구, 보험자부담률, 일인당_진료비_만원
FROM hira_model
ORDER BY 보험자부담률 DESC
LIMIT 5
"""
pd.read_sql(q, conn)

,시도,시군구,보험자부담률,일인당_진료비_만원
0,전남,화순군,84.9,280.6
1,울산,울산동구,80.4,156.2
2,서울,서대문구,80.1,150.0
3,경기,고양일산동구,79.5,127.4
4,부산,부산서구,79.1,189.6


> **보험자부담률 1위 화순군 84.9% — PY_02 해석 문장 확인** <br>
> 2~5위: 울산동구 80.4%, 서대문구 80.1%, 고양일산동구 79.5%, 부산서구 79.1%. <br>
> 화순군은 1인당 진료비(280.6만원, SQL_02 2.3-1)와 보험자부담률 모두 전국 1위.

#### 3.2-4 새 변수 3개 전체 행 대조 (SQL vs pandas)
- SQL_02 2.4-4와 같은 방식: PY_03·PY_04와 같은 식으로 pandas에서 다시 계산해 값이 다른 행 수를 센다

In [13]:
m = pd.read_sql("SELECT * FROM hira_model", conn)

metro = ['서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종']
py = pd.DataFrame({
    '광역시':       m['시도'].isin(metro).astype(int),
    '강원더미':     (m['시도'] == '강원').astype(int),
    '보험자부담률': (m['보험자부담금'] / m['요양급여비용총액'] * 100).round(1),
})
(m[py.columns] != py).sum()

광역시       0
강원더미      0
보험자부담률    0
dtype: int64

### 3.3 윈도우 함수 기초: 순위
- 윈도우 함수: 행을 묶어 줄이지 않고(251행 그대로), 각 행 옆에 "전체 또는 그룹 안에서의 계산값"을 붙여주는 함수. 형식은 `함수() OVER (…)`
- GROUP BY는 시도별로 묶어 17행으로 줄이지만, 윈도우 함수는 251행을 유지한 채 값을 붙인다
#### 3.3-1 순위 매기기와 동점 처리 방식 비교
- `RANK() OVER (ORDER BY 열)`: 작은 값부터 1, 2, 3… 순위. 같은 값이면 같은 순위(앞 번호)를 주고 다음 번호는 건너뛴다
- `COUNT(*) OVER (PARTITION BY 열)`: 같은 값을 가진 행끼리 칸막이를 치고, 그 칸 안의 행 수(=동점 수)를 각 행에 붙인다
- 평균 방식 순위 = RANK + (동점 수 − 1) ÷ 2 → 동점인 행들이 차지하는 순위의 가운데 값 (PY_03 `rankdata` 기본 방식)
- `ORDER BY 동점수 DESC`: 동점이 많은 값부터 보여줘서 두 방식이 갈리는 곳을 바로 확인

In [17]:
q = """
SELECT 시도, 시군구, 일인당_진료비_만원,
      RANK() OVER (ORDER BY 일인당_진료비_만원) AS 순위_RANK,
      COUNT(*) OVER (PARTITION BY 일인당_진료비_만원) AS 동점수,
      RANK() OVER (ORDER BY 일인당_진료비_만원)
        + (COUNT(*) OVER (PARTITION BY 일인당_진료비_만원) - 1 ) / 2.0 AS 순위_평균
FROM hira_model
ORDER BY 동점수 DESC, 일인당_진료비_만원
LIMIT 10
"""
pd.read_sql(q, conn)

,시도,시군구,일인당_진료비_만원,순위_RANK,동점수,순위_평균
0,서울,강북구,58.8,76,3,77.0
1,부산,부산연제구,58.8,76,3,77.0
2,전남,곡성군,58.8,76,3,77.0
3,서울,광진구,82.1,160,3,161.0
4,부산,부산기장군,82.1,160,3,161.0
5,광주,광주남구,82.1,160,3,161.0
6,전북,무주군,36.7,18,2,18.5
7,전북,진안군,36.7,18,2,18.5
8,경기,성남중원구,45.7,33,2,33.5
9,경북,울진군,45.7,33,2,33.5


> **동점 존재 확인 — 같은 값이 최대 3개** <br>
> 58.8만원(서울 강북구·부산연제구·전남 곡성군)과 82.1만원(서울 광진구·부산기장군·광주남구)이 각 3개, 36.7만원·45.7만원이 각 2개. <br>
> 58.8만원 3개: RANK 방식은 셋 다 76위, 평균 방식은 76·77·78위의 가운데인 77위. <br>
> 36.7만원 2개: RANK 방식 18위, 평균 방식 18.5위.

#### 3.3-2 시도별 평균 순위
- PY_03 [검정 3] 4. 시도별 평균 순위(`rankdata` 후 시도별 평균) 대응
- 윈도우 함수와 GROUP BY를 한 번에 쓸 수 없어서 서브쿼리로 단계를 나눈다: 안쪽에서 251행 각각에 순위를 붙이고, 바깥에서 시도별 평균
- 두 순위 방식을 나란히 계산해 동점 처리 차이가 결과에 주는 영향을 확인

In [14]:
q = """
SELECT 시도,
       ROUND(AVG(순위_RANK), 1) AS 평균순위_RANK,
       ROUND(AVG(순위_평균), 1) AS 평균순위_평균방식
FROM (
    SELECT 시도,
           RANK() OVER (ORDER BY 일인당_진료비_만원) AS 순위_RANK,
           RANK() OVER (ORDER BY 일인당_진료비_만원)
             + (COUNT(*) OVER (PARTITION BY 일인당_진료비_만원) - 1) / 2.0 AS 순위_평균
    FROM hira_model
)
GROUP BY 시도
ORDER BY 평균순위_평균방식 DESC
"""
pd.read_sql(q, conn)

,시도,평균순위_RANK,평균순위_평균방식
0,광주,179.8,180.1
1,부산,154.1,154.3
2,서울,150.8,151.0
3,전북,148.9,149.0
4,경남,147.6,147.8
5,인천,140.1,140.2
6,전남,140.0,140.1
7,울산,131.6,131.8
8,제주,131.0,131.3
9,대구,127.6,127.6


#### 3.3-3 [실험] 제주 시군구 순위 ① WHERE를 같은 쿼리에 넣은 경우
- 3.3-2에서 제주만 SQL 131.3 / PY_03 131.2로 0.1 차이 → 제주 시군구 2개의 순위를 직접 확인
- 윈도우 함수와 WHERE를 같은 쿼리에 쓰면 순위가 어떻게 계산되는지 실험

In [15]:
q = """
SELECT 시도, 시군구, 일인당_진료비_만원,
       RANK() OVER (ORDER BY 일인당_진료비_만원)
         + (COUNT(*) OVER (PARTITION BY 일인당_진료비_만원) - 1) / 2.0 AS 순위_평균
FROM hira_model
WHERE 시도 = '제주'
"""
pd.read_sql(q, conn)

,시도,시군구,일인당_진료비_만원,순위_평균
0,제주,서귀포시,51.9,1.0
1,제주,제주시,103.6,2.0


#### 3.3-4 [실험] 제주 시군구 순위 ② 서브쿼리로 순위를 먼저 매긴 뒤 WHERE
- 안쪽 쿼리에서 251개 전체 기준 순위를 붙이고, 바깥 쿼리에서 제주만 남긴다
- 3.3-3과 결과를 비교해 WHERE와 윈도우 함수의 실행 순서를 확인

In [16]:
q = """
SELECT 시도, 시군구, 일인당_진료비_만원, 순위_평균
FROM (
    SELECT 시도, 시군구, 일인당_진료비_만원,
           RANK() OVER (ORDER BY 일인당_진료비_만원)
             + (COUNT(*) OVER (PARTITION BY 일인당_진료비_만원) - 1) / 2.0 AS 순위_평균
    FROM hira_model
)
WHERE 시도 = '제주'
"""
pd.read_sql(q, conn)

,시도,시군구,일인당_진료비_만원,순위_평균
0,제주,서귀포시,51.9,47.5
1,제주,제주시,103.6,215.0
